# Rangka Kerja Ejen Microsoft — Azure OpenAI (API Respon)

Dalam contoh kod ini, anda akan menggunakan **Rangka Kerja Ejen Microsoft (MAF)** untuk mencipta ejen mudah yang disokong oleh **Azure OpenAI** menggunakan **API Respon**.

> **Nota migrasi:** Contoh ini sebelum ini menggunakan Semantic Kernel dengan Model GitHub. Ia telah dipindahkan ke Rangka Kerja Ejen Microsoft, dan Model GitHub (yang sudah usang, akan dihentikan pada Julai 2026) telah digantikan dengan Azure OpenAI, yang menyokong API Respon. `OpenAIChatClient` dalam MAF menyasarkan titik akhir stabil Azure OpenAI `/openai/v1/` dan menggunakan API Respon secara lalai.

Tujuan contoh ini adalah untuk menunjukkan langkah-langkah yang akan digunakan kemudian dalam contoh kod tambahan apabila melaksanakan pelbagai pola ejen.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Import Pakej Python yang Diperlukan


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Mendefinisikan Alat

Dalam Rangka Kerja Ejen Microsoft, **alat** adalah fungsi Python biasa yang dihiasi dengan `@tool` yang boleh dipanggil oleh ejen. Di bawah ini kami mendefinisikan alat yang mengembalikan destinasi percutian rawak dan mengelakkan pengulangan destinasi sebelumnya.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Mencipta Ejen

Di sini, kami mencipta Ejen bernama `TravelAgent`.

Dalam contoh ini, kami menggunakan arahan yang sangat asas. Sila ubah arahan ini untuk melihat bagaimana tingkah laku ejen berubah.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Menjalankan Ejen

Sekarang kita boleh menjalankan ejen. Kami mencipta `AgentSession` supaya ejen mengingati perbualan sepanjang giliran, kemudian menghantar dua `user_inputs`. Yang pertama meminta percutian; yang kedua mengatakan pengguna tidak menyukai cadangan itu dan meminta satu lagi — ejen menggunakan sejarah sesi ditambah alat `get_random_destination` untuk memberi respons.

Anda boleh mengubah mesej-mesej ini untuk melihat bagaimana ejen bertindak balas secara berbeza. Respons diberikan secara **aliran** token demi token.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
